# Lab 4.3 — Dual-stream training loop

This lab introduces the **dual-stream input** layout and the **structured attention mask** ($M_{BD} \cup M_{OBC} \cup M_{BC}$) from Lecture 2.2.

We jointly train one model on **both** the AR loss (next-token prediction with causal mask) and the diffusion loss (mask-then-denoise with block-diff mask). The two losses share weights, just like NLD-8B.

**Goals.**
1. Construct the 2L × 2L structured attention mask in PyTorch.
2. Run a single forward over `[noisy | clean]` and produce both losses.
3. Verify the model can do both AR generation and diffusion generation.

**Prerequisites.** Lab 4.1, Lab 4.2. Lectures 2.1, 2.2, 3.2.


In [1]:
# Cell 1 — imports and corpus
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

text = (
    "The quick brown fox jumps over the lazy dog. "
    "Pack my box with five dozen liquor jugs. "
    "How vexingly quick daft zebras jump! "
    "Sphinx of black quartz, judge my vow. "
    "The five boxing wizards jump quickly. "
) * 2000
chars = sorted(list(set(text))) + ["<MASK>"]
vocab_size = len(chars)
mask_token_id = vocab_size - 1
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] if i != mask_token_id else "_" for i in l)
data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
print(f"vocab={vocab_size}, mask_token_id={mask_token_id}")


Using device: cpu
vocab=35, mask_token_id=34


/home/ubuntu/.pyenv/versions/3.12.8/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. Constructing the structured attention mask

For a dual-stream of length $2L$ (noisy view positions $[0, L)$, clean view positions $[L, 2L)$), the structured mask is:

- **M_BD** (block-diagonal): positions in the same block AND same view can attend.
- **M_OBC** (offset-block-causal): noisy query in block $i$ can attend to clean key in blocks $< i$.
- **M_BC** (block-causal): clean query in block $i$ can attend to clean key in blocks $\leq i$.

Final allowed: $M_{BD} \cup M_{OBC} \cup M_{BC}$ — exactly as in NLD's `block_diff_mask` (Lecture 3.2 §3).


In [2]:
# Cell 2 — structured mask
def make_dual_stream_mask(L, block_size, device):
    '''Returns (2L, 2L) boolean mask where True means "can attend".'''
    total = 2 * L
    q = torch.arange(total, device=device).view(-1, 1)
    kv = torch.arange(total, device=device).view(1, -1)
    
    # x0_flag: 0 if noisy view, 1 if clean view
    x0_flag_q = (q >= L).long()
    x0_flag_kv = (kv >= L).long()
    
    # Block index within each view
    block_q = torch.where(x0_flag_q == 1, (q - L) // block_size, q // block_size)
    block_kv = torch.where(x0_flag_kv == 1, (kv - L) // block_size, kv // block_size)
    
    # M_BD: same block, same view
    M_BD = (block_q == block_kv) & (x0_flag_q == x0_flag_kv)
    
    # M_OBC: noisy q, clean kv, block_q > block_kv
    M_OBC = (block_q > block_kv) & (x0_flag_kv == 1) & (x0_flag_q == 0)
    
    # M_BC: clean q, clean kv, block_q >= block_kv
    M_BC = (block_q >= block_kv) & (x0_flag_kv == 1) & (x0_flag_q == 1)
    
    return M_BD | M_OBC | M_BC

# Sanity-check on a tiny example
mask_sanity = make_dual_stream_mask(L=8, block_size=4, device=device)
print(f"Mask shape: {mask_sanity.shape}")
print(f"True fraction: {mask_sanity.float().mean().item():.3f}")
# Visualize a tiny portion
print("\nMask (8 + 8, block_size=4):")
print(mask_sanity.int().cpu().numpy())


Mask shape: torch.Size([16, 16])
True fraction: 0.375

Mask (8 + 8, block_size=4):
[[1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 1 1 1 1 1 1 0 0 0 0]
 [0 0 0 0 1 1 1 1 1 1 1 1 0 0 0 0]
 [0 0 0 0 1 1 1 1 1 1 1 1 0 0 0 0]
 [0 0 0 0 1 1 1 1 1 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1]]


## 2. Model with configurable attention mode

The same TinyGPT backbone supports three modes via a `set_mode(mode)` call:
- `"autoregressive"` → causal mask (1L sequence).
- `"bidirectional"` → no mask (1L sequence).
- `"block_diff"` → structured 2L × 2L mask.


In [3]:
# Cell 3 — multi-mode model
@dataclass
class GPTConfig:
    vocab_size: int = 35
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    block_size: int = 256
    diff_block_size: int = 8

class FlexAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=False)

    def forward(self, x, attn_mask):
        '''attn_mask: (T, T) boolean, True = allowed.'''
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        att = att.masked_fill(~attn_mask.unsqueeze(0).unsqueeze(0), float("-inf"))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)
    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attn = FlexAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    def forward(self, x, attn_mask):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.mlp(self.ln2(x))
        return x

class TinyJointModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
    
    def forward(self, idx, attn_mask):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x, attn_mask)
        return self.head(self.ln_f(x))

config = GPTConfig(vocab_size=vocab_size, n_layer=4, n_head=4, n_embd=128,
                    block_size=256, diff_block_size=8)
model = TinyJointModel(config).to(device)
print(f"Total params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


Total params: 0.83M


## 3. The joint training step

For each batch:
1. Sample mask ratio, construct `noisy_ids`.
2. Concatenate `[noisy_ids | clean_ids]` → `(B, 2L)`.
3. Forward with the **block_diff mask** → logits at all 2L positions.
4. **Diffusion loss** = CE at masked positions in the noisy half.
5. **AR loss** = CE at all positions in the clean half (shifted-by-one prediction). Computed using the **causal mask** in a separate forward.
6. Total loss = L_AR + α · L_diff, with α = 0.5.


In [4]:
# Cell 4 — joint training step
def make_causal_mask(L, device):
    return torch.tril(torch.ones(L, L, dtype=torch.bool, device=device))

def joint_training_step(model, x, mask_token_id, block_size_diff,
                         alpha=0.5, eps=0.01):
    B, L = x.shape
    rho = eps + (1 - 2*eps) * torch.rand(B, 1, device=x.device)
    mask_indices = torch.rand(B, L, device=x.device) < rho
    if not mask_indices.any():
        mask_indices[0, 0] = True
    noisy = torch.where(mask_indices, mask_token_id, x)
    
    # AR forward (causal, length L)
    causal = make_causal_mask(L, device=x.device)
    ar_logits = model(x, causal)
    L_ar = F.cross_entropy(ar_logits[:, :-1].reshape(-1, ar_logits.size(-1)),
                            x[:, 1:].reshape(-1))
    
    # Diffusion forward (block_diff, length 2L)
    dual = torch.cat([noisy, x], dim=-1)
    bd_mask = make_dual_stream_mask(L, block_size_diff, device=x.device)
    bd_logits = model(dual, bd_mask)
    noisy_logits = bd_logits[:, :L]
    L_diff = F.cross_entropy(noisy_logits[mask_indices].view(-1, noisy_logits.size(-1)),
                              x[mask_indices].view(-1))
    
    return L_ar, L_diff, L_ar + alpha * L_diff

def get_batch(split, batch_size=16, seq_len=64):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - seq_len, (batch_size,))
    return torch.stack([src[i:i+seq_len] for i in ix]).to(device)

@torch.no_grad()
def estimate_losses(eval_iters=10):
    model.eval()
    out = {"L_ar": [], "L_diff": []}
    for _ in range(eval_iters):
        x = get_batch("val")
        ar, diff, _ = joint_training_step(model, x, mask_token_id,
                                            config.diff_block_size)
        out["L_ar"].append(ar.item())
        out["L_diff"].append(diff.item())
    model.train()
    return {k: sum(v)/len(v) for k, v in out.items()}

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
steps = 150
for step in range(steps):
    x = get_batch("train")
    L_ar, L_diff, loss = joint_training_step(model, x, mask_token_id,
                                                config.diff_block_size)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 30 == 0 or step == steps - 1:
        l = estimate_losses()
        print(f"step {step:4d}  L_ar {l['L_ar']:.3f}  L_diff {l['L_diff']:.3f}  total {loss.item():.3f}")
print("\n--- Joint training complete ---")


step    0  L_ar 3.077  L_diff 3.384  total 5.686


step   30  L_ar 0.950  L_diff 3.157  total 2.547


step   60  L_ar 0.177  L_diff 3.019  total 1.685


step   90  L_ar 0.081  L_diff 2.575  total 1.419


step  120  L_ar 0.080  L_diff 1.982  total 1.131


step  149  L_ar 0.066  L_diff 1.570  total 0.878

--- Joint training complete ---


## 4. AR generation (using the joint model)

The joint model can act as a pure AR LM using the causal mask. We verify it generates plausible continuations.


In [5]:
# Cell 5 — AR generation
@torch.no_grad()
def ar_generate(model, prompt: str, max_new_tokens: int = 60):
    model.eval()
    ids = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    block_size = config.block_size
    for _ in range(max_new_tokens):
        idx = ids[:, -block_size:]
        causal = make_causal_mask(idx.shape[1], device=device)
        logits = model(idx, causal)
        next_id = logits[0, -1].argmax().unsqueeze(0).unsqueeze(0)
        ids = torch.cat([ids, next_id], dim=1)
    return decode(ids[0].tolist())

print("AR generation:")
print(ar_generate(model, "The quick", max_new_tokens=60))


AR generation:
The quick brown fox jumps over the lazy dog. Pack my box with five fi


## 5. Block-diffusion generation (using the joint model)

The same model can also do diffusion drafting. We use the dual-stream layout at inference: the prompt acts as the clean prefix; the masked region acts as the noisy view.

For simplicity here, we use a single forward and threshold-based commit (Lecture 1.5 §3).


In [6]:
# Cell 6 — block-diffusion generation
@torch.no_grad()
def block_diffusion_generate(model, prompt: str, gen_length: int = 32,
                               block_length: int = 8, threshold: float = 0.5):
    model.eval()
    prompt_ids = torch.tensor(encode(prompt), dtype=torch.long, device=device)
    L_p = len(prompt_ids)
    
    # Pad clean side to a multiple of block_length for clean mask construction
    committed = prompt_ids.clone()
    nfe = 0
    remaining = gen_length
    while remaining > 0:
        block = min(block_length, remaining)
        # Construct dual stream: pad to L = current commit + block_length
        L = committed.shape[0] + block
        # Noisy view: pad clean + mask the next `block` positions
        noisy = torch.cat([committed,
                           torch.full((block,), mask_token_id, device=device)])
        clean = noisy.clone()  # placeholder — will be filled by model
        dual = torch.cat([noisy, clean]).unsqueeze(0)
        mask = make_dual_stream_mask(L, block_length, device)
        logits = model(dual, mask)
        nfe += 1
        # Read predictions from the noisy view at the masked positions
        probs = F.softmax(logits[0, :L], dim=-1)
        block_start = committed.shape[0]
        block_probs = probs[block_start:block_start + block]
        max_p, argmax = block_probs.max(dim=-1)
        # Commit all positions above threshold (or all if none above)
        accept = max_p >= threshold
        if not accept.any():
            accept[0] = True  # commit at least one
        # Take longest accepted prefix
        n_accept = 0
        for i in range(block):
            if accept[i]:
                n_accept += 1
            else:
                break
        if n_accept == 0:
            n_accept = 1
        committed = torch.cat([committed, argmax[:n_accept]])
        remaining -= n_accept
    return decode(committed.tolist()), nfe

generated, nfe = block_diffusion_generate(model, "The quick", gen_length=32,
                                            block_length=8, threshold=0.3)
print(generated)
print(f"\nNFE = {nfe}, gen_length = 32, TPF = {32/nfe:.2f}")


The quick bbbbbb oooxxrxrrxx  wnnnnnxnn o

NFE = 6, gen_length = 32, TPF = 5.33


## 6. Sanity checks

Quick assertions to verify mask structure and the dual-stream forward.


In [7]:
# Cell 7 — assertions
mask = make_dual_stream_mask(L=16, block_size=4, device=device)

# Diagonal must be True
assert mask.diagonal().all(), "self-attention should be allowed"

# Noisy → noisy across blocks must be False
assert not mask[0, 5], "noisy block 0 -> noisy block 1 should be forbidden"

# Noisy → clean within own block must be False (label leak guard)
assert not mask[0, 16], "noisy block 0 -> clean block 0 should be forbidden"

# Noisy → clean previous block must be True (M_OBC)
assert mask[5, 16], "noisy block 1 -> clean block 0 should be allowed"

# Clean → noisy must be False
assert not mask[16, 0], "clean -> noisy is always forbidden"

# Clean → future clean must be False
assert not mask[16, 20], "clean block 0 -> clean block 1 forbidden"

# Clean → past clean must be True
assert mask[20, 16], "clean block 1 -> clean block 0 allowed"

print("All mask assertions passed.")

# Forward shape check
x_test = get_batch("train")
B, L = x_test.shape
dual = torch.cat([x_test, x_test], dim=-1)
bd_mask = make_dual_stream_mask(L, config.diff_block_size, device=device)
logits = model(dual, bd_mask)
assert logits.shape == (B, 2*L, vocab_size), f"got {logits.shape}"
print(f"Dual-stream forward shape OK: {logits.shape}")


All mask assertions passed.
Dual-stream forward shape OK: torch.Size([16, 128, 35])


## 7. Recap

We have a **joint AR + diffusion** model that:
- Shares weights between AR mode and diffusion mode.
- Trains with **both losses simultaneously** ($L = L_{AR} + 0.5 \cdot L_{diff}$).
- Generates in **either** mode at inference time.
- Uses the structured 2L × 2L attention mask (M_BD ∪ M_OBC ∪ M_BC).

This is the same machinery as NLD-8B, scaled down to 1M parameters and a tiny corpus.

Next lab (4.4): implement a **mode-switching** `generate()` that uses both modes in tandem — diffusion drafting + AR verification = self-speculation.
